# Smoke page: loading one FRESH recording with Cedalion

This is a smoke page: it exists to test the build machinery, it is not a chapter, and it is removed from the table of contents before version 1.0.

## Objectives, prerequisites and reading time

- Objective: check that the book builds, that a notebook executes headless, and that Cedalion reads a FRESH recording through the path helper of the project.
- Prerequisites: none; the page teaches no fNIRS concept.
- Estimated reading time: one minute.

## Concept

This section is a placeholder that exercises the second part of the chapter template. A real chapter states one concept here, with one or two figures.

## Hands-on

The next cells read one motor recording of the FRESH dataset, report its size and sampling rate, and plot two seconds of raw intensity on one short and one long channel.

In [ ]:
import cedalion
import cedalion.io
import cedalion.nirs
import matplotlib.pyplot as plt
import numpy as np

from fnirs_tutorials import paths, style

style.apply_style()

In [ ]:
# The path is built from the data directory of the project, never from an absolute
# path.
snirf_path = (
    paths.data_dir()
    / "fresh"
    / "FRESH-Motor"
    / "sub-01"
    / "ses-left2s"
    / "nirs"
    / "sub-01_ses-left2s_task-FRESHMOTOR_nirs.snirf"
)

# read_snirf returns a list of Recording objects, one per SNIRF data block; this file
# has one.
recording = cedalion.io.read_snirf(snirf_path)[0]

# 'amp' is the raw intensity (amplitude) time series, with dimensions (channel,
# wavelength, time). The file declares no unit for the measurement, so Cedalion
# attaches a dimensionless placeholder; dequantify() moves it to an attribute so that
# plain NumPy access below does not warn about a stripped unit.
amplitudes = recording.timeseries["amp"].pint.dequantify()

# Both numbers are read from the loaded recording, never typed in.
n_channels = amplitudes.sizes["channel"]
sampling_rate_hz = 1.0 / float(np.diff(amplitudes.time.values).mean())
print(f"Channels (source-detector pairs): {n_channels}")
print(f"Wavelengths (nm): {amplitudes.wavelength.values.tolist()}")
print(f"Sampling rate: {sampling_rate_hz:.2f} Hz")

In [ ]:
# Illustration constants. The pick below only serves this illustration: choosing the
# channel closest to the target distance as the long channel is a heuristic, not a
# methodological recommendation.
LONG_CHANNEL_TARGET_CM = 3.0  # source-detector distance sought for the long channel
WINDOW_S = 2.0  # duration of the plotted window, from the start of the recording

# channel_distances gives the source-detector distance of every channel, in the length
# unit of the probe geometry (metres in this file); .pint.to("cm") converts it.
distances_cm = (
    cedalion.nirs.channel_distances(amplitudes, recording.geo3d)
    .pint.to("cm")
    .pint.dequantify()
)

short_channel = str(distances_cm.idxmin("channel").item())
long_channel = str(
    np.abs(distances_cm - LONG_CHANNEL_TARGET_CM).idxmin("channel").item()
)
if short_channel == long_channel:
    raise ValueError(
        f"The shortest channel and the channel closest to {LONG_CHANNEL_TARGET_CM} cm "
        f"are the same channel ({short_channel}); this recording has no distinct pair "
        "to compare."
    )

window = amplitudes.sel(time=slice(0.0, WINDOW_S))

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
for ax, channel in zip(axes, [short_channel, long_channel], strict=True):
    distance = float(distances_cm.sel(channel=channel))
    for wavelength in window.wavelength.values:
        trace = window.sel(channel=channel, wavelength=wavelength)
        ax.plot(
            trace.time.values,
            trace.values,
            label=f"{channel}, {wavelength:.0f} nm, {distance:.2f} cm",
        )
    ax.set_ylabel("Raw intensity (a.u.)")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    "Raw intensity on the shortest channel and on the channel closest to "
    f"{LONG_CHANNEL_TARGET_CM:g} cm"
)
plt.show()

## Pitfall demonstration

This section is a placeholder that exercises the fourth part of the chapter template. A real chapter shows here, on data, one way the analysis goes wrong.

## In the field / in LIONirs

This box is a placeholder that exercises the fifth part of the chapter template.

```{note} In the field / in LIONirs
A real chapter says here how the same step is done in the field and in the LIONirs toolbox.
```

## Self-check and further reading

This section is a placeholder that exercises the sixth part of the chapter template. A real chapter ends with three to five questions and a short list of verified references.